#  Llama 3.1 8B Inference Engine — From Scratch in C++

This notebook implements a **complete** Llama 3.1 8B inference engine using the **ggml** tensor library.

**What it does:**
1. **Phase 1**: Loads a GGUF model file (weights, vocab, architecture metadata)
2. **Phase 2**: Prefills the KV cache by processing your entire prompt at once
3. **Phase 3**: Generates tokens one-by-one in a decode loop until `<|eot_id|>`

**Architecture implemented:**
- RMSNorm (pre-norm)
- Rotary Positional Embeddings (RoPE, θ=500,000)
- Grouped-Query Attention (32 Q heads, 8 KV heads)
- SwiGLU Feed-Forward Network
- KV Cache for efficient autoregressive generation


## Step 1: Install Dependencies & Clone ggml

We need the ggml library (the tensor computation engine). We'll build it from source.

In [128]:
%%shell
# Check for g++ and cmake
echo "=== Checking build tools ==="
g++ --version | head -1
cmake --version | head -1

# Clone ggml
echo "=== Cloning ggml ==="
if [ ! -d "ggml" ]; then
    git clone --depth 1 https://github.com/ggml-org/ggml.git
    echo "ggml cloned."
else
    echo "ggml already present."
fi

# Build ggml as a static library
echo "=== Building ggml ==="
cd ggml
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DGGML_BUILD_TESTS=OFF -DGGML_BUILD_EXAMPLES=OFF -DGGML_CUDA=ON
make -j$(nproc)
echo "=== ggml built ==="
ls -la src/libggml*.a src/libggml*.so 2>/dev/null || echo "(checking lib locations...)"
find . -name "libggml*" -type f | head -20


=== Checking build tools ===
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
cmake version 3.31.10
=== Cloning ggml ===
ggml already present.
=== Building ggml ===
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75-real CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- CUDA host compiler is GNU 11.4.0
-- Including CUDA backend
-- ggml version: 0.9.8
-- ggml commit:  c044a8e
-- Configuring done (0.3s)
-- Generating done (0.1s)
-- Build files have been written to: /content/ggml/build
[  6%] Built target ggml-base
[ 16%] Built target ggml-cpu
[ 99%] Built target ggml-cuda
[100%] Built target ggml
=== ggml built ===
lrwxrwxrwx 1 root root 17 Mar 26 05:22  src/libggml-base.so -> libggml-base.so.0
lrwxrwxrwx 1 root root 16

## Step 2: Write C++ Source Files

These cells use `%%writefile` to create all the source files for our inference engine.

In [129]:
%%shell
mkdir -p src


### `gguf_loader.h`

In [130]:
%%writefile src/gguf_loader.h
#pragma once
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>
#include <unordered_map>
#include <stdexcept>

enum GGUFMetadataValueType : uint32_t {
    GGUF_TYPE_UINT8   = 0,
    GGUF_TYPE_INT8    = 1,
    GGUF_TYPE_UINT16  = 2,
    GGUF_TYPE_INT16   = 3,
    GGUF_TYPE_UINT32  = 4,
    GGUF_TYPE_INT32   = 5,
    GGUF_TYPE_FLOAT32 = 6,
    GGUF_TYPE_BOOL    = 7,
    GGUF_TYPE_STRING  = 8,
    GGUF_TYPE_ARRAY   = 9,
    GGUF_TYPE_UINT64  = 10,
    GGUF_TYPE_INT64   = 11,
    GGUF_TYPE_FLOAT64 = 12,
};

enum GGUF_TYPE_ID : uint32_t {
    GGUF_TID_F32     = 0,
    GGUF_TID_F16     = 1,
    GGUF_TID_Q4_0    = 2,
    GGUF_TID_Q4_1    = 3,
    GGUF_TID_Q5_0    = 6,
    GGUF_TID_Q5_1    = 7,
    GGUF_TID_Q8_0    = 8,
    GGUF_TID_Q8_1    = 9,
    GGUF_TID_Q2_K    = 10,
    GGUF_TID_Q3_K    = 11,
    GGUF_TID_Q4_K    = 12,
    GGUF_TID_Q5_K    = 13,
    GGUF_TID_Q6_K    = 14,
    GGUF_TID_Q8_K    = 15,
    GGUF_TID_IQ2_XXS = 16,
    GGUF_TID_IQ2_XS  = 17,
    GGUF_TID_IQ3_XXS = 18,
    GGUF_TID_IQ1_S   = 19,
    GGUF_TID_IQ4_NL  = 20,
    GGUF_TID_IQ3_S   = 21,
    GGUF_TID_IQ2_S   = 22,
    GGUF_TID_IQ4_XS  = 23,
    GGUF_TID_I8      = 24,
    GGUF_TID_I16     = 25,
    GGUF_TID_I32     = 26,
    GGUF_TID_I64     = 27,
    GGUF_TID_F64     = 28,
    GGUF_TID_BF16    = 30,
};

struct GGUFTensorInfo {
    std::string name;
    uint32_t    n_dims;
    uint64_t    dims[4];
    GGUF_TYPE_ID type;
    uint64_t    offset;
};

struct GGUFValue {
    GGUFMetadataValueType type;
    union {
        uint8_t  u8;
        int8_t   i8;
        uint16_t u16;
        int16_t  i16;
        uint32_t u32;
        int32_t  i32;
        float    f32;
        bool     b;
        uint64_t u64;
        int64_t  i64;
        double   f64;
    };
    std::string              str;
    std::vector<GGUFValue>   arr;
    GGUFMetadataValueType    arr_type;
};

struct GGUFFile {
    uint32_t version;
    uint64_t tensor_count;
    uint64_t metadata_kv_count;
    std::unordered_map<std::string, GGUFValue> metadata;
    std::vector<GGUFTensorInfo> tensors;
    uint64_t tensor_data_offset;
    FILE* fp = nullptr;
    ~GGUFFile() { if (fp) fclose(fp); }
    uint32_t get_u32(const std::string& key) const { return metadata.at(key).u32; }
    uint64_t get_u64(const std::string& key) const { return metadata.at(key).u64; }
    int32_t get_i32(const std::string& key) const { return metadata.at(key).i32; }
    float get_f32(const std::string& key) const { return metadata.at(key).f32; }
    float get_f32_or(const std::string& key, float def) const { return metadata.count(key) ? metadata.at(key).f32 : def; }
    uint32_t get_u32_or(const std::string& key, uint32_t def) const { return metadata.count(key) ? metadata.at(key).u32 : def; }
    const std::string& get_str(const std::string& key) const { return metadata.at(key).str; }
    const std::vector<GGUFValue>& get_arr(const std::string& key) const { return metadata.at(key).arr; }
    bool has_key(const std::string& key) const { return metadata.find(key) != metadata.end(); }
};

GGUFFile load_gguf(const char* path);
void read_tensor_data(GGUFFile& gguf, const GGUFTensorInfo& info, void* dst, size_t nbytes);
size_t gguf_tensor_nbytes(const GGUFTensorInfo& info);

Overwriting src/gguf_loader.h


### `gguf_loader.cpp`

In [131]:
%%writefile src/gguf_loader.cpp
#include "gguf_loader.h"
#include <algorithm>

static inline void read_bytes(FILE* fp, void* dst, size_t n) {
    if (fread(dst, 1, n, fp) != n) throw std::runtime_error("GGUF: unexpected end of file");
}

template<typename T> static inline T read_val(FILE* fp) {
    T val; read_bytes(fp, &val, sizeof(T)); return val;
}

static std::string read_gguf_string(FILE* fp) {
    uint64_t len = read_val<uint64_t>(fp);
    std::string s(len, '\0');
    read_bytes(fp, s.data(), len);
    return s;
}

static void get_type_info(GGUF_TYPE_ID type, size_t& block_size, size_t& type_size) {
    switch (type) {
        case GGUF_TID_F32:     block_size = 1;   type_size = 4;  break;
        case GGUF_TID_F16:     block_size = 1;   type_size = 2;  break;
        case GGUF_TID_Q4_0:    block_size = 32;  type_size = 18; break;
        case GGUF_TID_Q4_1:    block_size = 32;  type_size = 20; break;
        case GGUF_TID_Q5_0:    block_size = 32;  type_size = 22; break;
        case GGUF_TID_Q5_1:    block_size = 32;  type_size = 24; break;
        case GGUF_TID_Q8_0:    block_size = 32;  type_size = 34; break;
        case GGUF_TID_Q8_1:    block_size = 32;  type_size = 36; break;
        case GGUF_TID_Q2_K:    block_size = 256; type_size = 84; break;
        case GGUF_TID_Q3_K:    block_size = 256; type_size = 110; break;
        case GGUF_TID_Q4_K:    block_size = 256; type_size = 144; break;
        case GGUF_TID_Q5_K:    block_size = 256; type_size = 176; break;
        case GGUF_TID_Q6_K:    block_size = 256; type_size = 210; break;
        case GGUF_TID_Q8_K:    block_size = 256; type_size = 292; break;
        case GGUF_TID_BF16:    block_size = 1;   type_size = 2;  break;
        case GGUF_TID_I8:      block_size = 1;   type_size = 1;  break;
        case GGUF_TID_I16:     block_size = 1;   type_size = 2;  break;
        case GGUF_TID_I32:     block_size = 1;   type_size = 4;  break;
        case GGUF_TID_I64:     block_size = 1;   type_size = 8;  break;
        case GGUF_TID_F64:     block_size = 1;   type_size = 8;  break;
        default: throw std::runtime_error("GGUF: unsupported tensor type");
    }
}

size_t gguf_tensor_nbytes(const GGUFTensorInfo& info) {
    uint64_t n_elements = 1;
    for (uint32_t i = 0; i < info.n_dims; i++) n_elements *= info.dims[i];
    size_t block_size, type_size;
    get_type_info(info.type, block_size, type_size);
    return (n_elements / block_size) * type_size;
}

static GGUFValue read_metadata_value(FILE* fp, GGUFMetadataValueType type) {
    GGUFValue val; val.type = type;
    switch (type) {
        case GGUF_TYPE_UINT8:   val.u8  = read_val<uint8_t>(fp);  break;
        case GGUF_TYPE_INT8:    val.i8  = read_val<int8_t>(fp);   break;
        case GGUF_TYPE_UINT16:  val.u16 = read_val<uint16_t>(fp); break;
        case GGUF_TYPE_INT16:   val.i16 = read_val<int16_t>(fp);  break;
        case GGUF_TYPE_UINT32:  val.u32 = read_val<uint32_t>(fp); break;
        case GGUF_TYPE_INT32:   val.i32 = read_val<int32_t>(fp);  break;
        case GGUF_TYPE_FLOAT32: val.f32 = read_val<float>(fp);    break;
        case GGUF_TYPE_BOOL:    val.b   = read_val<uint8_t>(fp);  break;
        case GGUF_TYPE_STRING:  val.str = read_gguf_string(fp);   break;
        case GGUF_TYPE_UINT64:  val.u64 = read_val<uint64_t>(fp); break;
        case GGUF_TYPE_INT64:   val.i64 = read_val<int64_t>(fp);  break;
        case GGUF_TYPE_FLOAT64: val.f64 = read_val<double>(fp);   break;
        case GGUF_TYPE_ARRAY: {
            GGUFMetadataValueType arr_type = read_val<GGUFMetadataValueType>(fp);
            uint64_t arr_len = read_val<uint64_t>(fp);
            val.arr_type = arr_type;
            val.arr.resize(arr_len);
            for (uint64_t i = 0; i < arr_len; i++) val.arr[i] = read_metadata_value(fp, arr_type);
            break;
        }
        default: throw std::runtime_error("GGUF: unknown metadata type");
    }
    return val;
}

GGUFFile load_gguf(const char* path) {
    GGUFFile gguf; FILE* fp = fopen(path, "rb");
    if (!fp) throw std::runtime_error("Cannot open GGUF file");
    gguf.fp = fp;
    uint32_t magic = read_val<uint32_t>(fp);
    if (magic != 0x46554747) {
        char buf[5] = {0};
        memcpy(buf, &magic, 4);
        printf("[Debug] Magic Read: 0x%08X ('%s')\n", magic, buf);
        throw std::runtime_error("GGUF: invalid magic");
    }
    gguf.version = read_val<uint32_t>(fp);
    gguf.tensor_count = read_val<uint64_t>(fp);
    gguf.metadata_kv_count = read_val<uint64_t>(fp);
    printf("[Debug] GGUF Version: %u, Tensors: %lu, Metadata KV: %lu\n", gguf.version, gguf.tensor_count, gguf.metadata_kv_count);
    for (uint64_t i = 0; i < gguf.metadata_kv_count; i++) {
        std::string key = read_gguf_string(fp);
        GGUFMetadataValueType vtype = read_val<GGUFMetadataValueType>(fp);
        gguf.metadata[key] = read_metadata_value(fp, vtype);
    }
    gguf.tensors.resize(gguf.tensor_count);
    for (uint64_t i = 0; i < gguf.tensor_count; i++) {
        auto& t = gguf.tensors[i];
        t.name = read_gguf_string(fp);
        t.n_dims = read_val<uint32_t>(fp);
        for (uint32_t d = 0; d < t.n_dims; d++) t.dims[d] = read_val<uint64_t>(fp);
        t.type = (GGUF_TYPE_ID)read_val<uint32_t>(fp);
        t.offset = read_val<uint64_t>(fp);
    }
    uint32_t align = gguf.get_u32_or("general.alignment", 32);
    gguf.tensor_data_offset = (ftell(fp) + align - 1) & ~(uint64_t)(align - 1);
    printf("[Debug] Tensor data offset: %lu\n", gguf.tensor_data_offset);
    return gguf;
}

void read_tensor_data(GGUFFile& gguf, const GGUFTensorInfo& info, void* dst, size_t nbytes) {
    fseek(gguf.fp, gguf.tensor_data_offset + info.offset, SEEK_SET);
    if (fread(dst, 1, nbytes, gguf.fp) != nbytes) throw std::runtime_error("GGUF: read fail");
}

Overwriting src/gguf_loader.cpp


### `tokenizer.h`

In [132]:
%%writefile src/tokenizer.h
#pragma once
#include "gguf_loader.h"
#include <string>
#include <vector>
#include <unordered_map>
#include <utility>

// =============================================================================
// BPE Tokenizer for Llama 3.1
// Loads vocab + merges from GGUF metadata, encodes/decodes text
// =============================================================================

struct Tokenizer {
    // Vocabulary: token_id -> token string (byte-level BPE pieces)
    std::vector<std::string> vocab;

    // Reverse mapping: token string -> token_id
    std::unordered_map<std::string, int32_t> token_to_id;

    // Token types (normal, unknown, control, user_defined, byte, etc.)
    std::vector<int32_t> token_types;

    // Token scores (used for some tokenizer variants)
    std::vector<float> token_scores;

    // BPE merge pairs: (piece_a, piece_b) in priority order
    std::vector<std::pair<std::string, std::string>> merges;

    // Merge rank: "piece_a piece_b" -> rank (lower = higher priority)
    std::unordered_map<std::string, int> merge_ranks;

    // Special token IDs
    int32_t bos_token_id = -1;     // <|begin_of_text|>
    int32_t eos_token_id = -1;     // <|end_of_text|>
    int32_t eot_token_id = -1;     // <|eot_id|>

    // Llama 3 special tokens (looked up by string)
    int32_t start_header_id = -1;  // <|start_header_id|>
    int32_t end_header_id   = -1;  // <|end_header_id|>
    int32_t nl_token_id     = -1;  // newline token

    int32_t vocab_size = 0;

    // Load from GGUF metadata
    void load(const GGUFFile& gguf);

    // Encode text to token IDs
    std::vector<int32_t> encode(const std::string& text) const;

    // Decode a single token ID to string
    std::string decode(int32_t token_id) const;

    // Decode a sequence of token IDs to string
    std::string decode(const std::vector<int32_t>& tokens) const;

    // Format a chat prompt using Llama 3.1 template
    std::string format_chat(const std::string& system_prompt,
                            const std::string& user_prompt) const;

    // Get the token IDs for a formatted chat prompt (with BOS)
    std::vector<int32_t> encode_chat(const std::string& system_prompt,
                                     const std::string& user_prompt) const;
};


Overwriting src/tokenizer.h


### `tokenizer.cpp`

In [133]:
%%writefile src/tokenizer.cpp
#include "tokenizer.h"
#include <algorithm>
#include <sstream>
#include <limits>
#include <cassert>
#include <climits>

void Tokenizer::load(const GGUFFile& gguf) {
    const auto& tokens_arr = gguf.get_arr("tokenizer.ggml.tokens");
    vocab_size = (int32_t)tokens_arr.size();
    vocab.resize(vocab_size);
    for (int32_t i = 0; i < vocab_size; i++) {
        vocab[i] = tokens_arr[i].str;
        token_to_id[vocab[i]] = i;
    }

    // Load token scores if available
    if (gguf.has_key("tokenizer.ggml.scores")) {
        const auto& scores_arr = gguf.get_arr("tokenizer.ggml.scores");
        token_scores.resize(vocab_size);
        for (int32_t i = 0; i < vocab_size; i++) {
            token_scores[i] = scores_arr[i].f32;
        }
    }

    // Load BPE merges if available
    if (gguf.has_key("tokenizer.ggml.merges")) {
        const auto& merges_arr = gguf.get_arr("tokenizer.ggml.merges");
        merges.resize(merges_arr.size());
        for (size_t i = 0; i < merges_arr.size(); i++) {
            const std::string& m = merges_arr[i].str;
            size_t sp = m.find(' ');
            if (sp != std::string::npos) {
                merges[i] = {m.substr(0, sp), m.substr(sp + 1)};
                merge_ranks[m] = (int)i;
            }
        }
        printf("[Tokenizer] Loaded %zu BPE merges\n", merges.size());
    } else {
        printf("[Tokenizer] No BPE merges found, using greedy matching\n");
    }

    // Debug: check if single ASCII chars are in vocab
    int ascii_found = 0;
    for (int c = 32; c < 127; c++) {
        std::string s(1, (char)c);
        if (token_to_id.count(s)) ascii_found++;
    }
    printf("[Tokenizer] ASCII chars in vocab: %d/95\n", ascii_found);

    auto find_tok = [&](const std::string& s) -> int32_t {
        auto it = token_to_id.find(s);
        return (it != token_to_id.end()) ? it->second : -1;
    };

    bos_token_id = (int32_t)gguf.get_u32_or("tokenizer.ggml.bos_token_id", find_tok("<|begin_of_text|>"));
    eos_token_id = (int32_t)gguf.get_u32_or("tokenizer.ggml.eos_token_id", find_tok("<|end_of_text|>"));
    eot_token_id = find_tok("<|eot_id|>");
    start_header_id = find_tok("<|start_header_id|>");
    end_header_id = find_tok("<|end_header_id|>");

    // Debug: show some sample tokens
    printf("[Tokenizer] Sample vocab: [0]='%s' [1]='%s' [2]='%s'\n",
           vocab[0].c_str(), vocab[1].c_str(), vocab[2].c_str());
    int32_t h_id = find_tok("H");
    int32_t e_id = find_tok("e");
    printf("[Tokenizer] Lookup: 'H'=%d, 'e'=%d\n", h_id, e_id);

    printf("[Tokenizer] Loaded vocab (size=%d). Special IDs: BOS=%d, EOS=%d, EOT=%d\n",
           vocab_size, bos_token_id, eos_token_id, eot_token_id);
}

// BPE encode: start with individual characters, then merge using merge_ranks
static std::vector<std::string> bpe_merge(const std::vector<std::string>& initial_tokens,
                                           const std::unordered_map<std::string, int>& merge_ranks) {
    std::vector<std::string> tokens = initial_tokens;
    while (tokens.size() >= 2) {
        int best_rank = INT_MAX;
        size_t best_idx = 0;
        for (size_t i = 0; i + 1 < tokens.size(); i++) {
            std::string pair_key = tokens[i] + " " + tokens[i + 1];
            auto it = merge_ranks.find(pair_key);
            if (it != merge_ranks.end() && it->second < best_rank) {
                best_rank = it->second;
                best_idx = i;
            }
        }
        if (best_rank == INT_MAX) break;
        tokens[best_idx] = tokens[best_idx] + tokens[best_idx + 1];
        tokens.erase(tokens.begin() + best_idx + 1);
    }
    return tokens;
}

std::vector<int32_t> Tokenizer::encode(const std::string& text) const {
    if (text.empty()) return {};

    std::vector<int32_t> ids;

    if (!merge_ranks.empty()) {
        // BPE encoding: start with individual bytes, then merge
        std::vector<std::string> initial_tokens;
        for (size_t i = 0; i < text.size(); i++) {
            initial_tokens.push_back(std::string(1, text[i]));
        }
        auto merged = bpe_merge(initial_tokens, merge_ranks);
        for (const auto& tok : merged) {
            auto it = token_to_id.find(tok);
            if (it != token_to_id.end()) {
                ids.push_back(it->second);
            } else {
                // Byte fallback for unrecognized pieces
                for (unsigned char c : tok) {
                    char buf[16];
                    snprintf(buf, sizeof(buf), "<0x%02X>", c);
                    auto bit = token_to_id.find(std::string(buf));
                    if (bit != token_to_id.end()) {
                        ids.push_back(bit->second);
                    }
                }
            }
        }
    } else {
        // Greedy longest-match fallback (no merges available)
        size_t i = 0;
        while (i < text.size()) {
            int best_len = 0;
            int32_t best_id = -1;
            int max_len = std::min((int)(text.size() - i), 64);
            for (int len = max_len; len >= 1; len--) {
                std::string sub = text.substr(i, len);
                auto it = token_to_id.find(sub);
                if (it != token_to_id.end()) {
                    best_len = len;
                    best_id = it->second;
                    break;
                }
            }
            if (best_id >= 0) {
                ids.push_back(best_id);
                i += best_len;
            } else {
                // Byte fallback
                char buf[16];
                snprintf(buf, sizeof(buf), "<0x%02X>", (unsigned char)text[i]);
                auto it = token_to_id.find(std::string(buf));
                if (it != token_to_id.end()) {
                    ids.push_back(it->second);
                } else {
                    printf("[Tokenizer] WARNING: byte 0x%02X not found in vocab\n", (unsigned char)text[i]);
                }
                i++;
            }
        }
    }
    return ids;
}

std::string Tokenizer::decode(int32_t token_id) const {
    if (token_id < 0 || token_id >= vocab_size) return "";
    return vocab[token_id];
}

std::string Tokenizer::decode(const std::vector<int32_t>& tokens) const {
    std::string res; for (int32_t t : tokens) res += decode(t); return res;
}

std::vector<int32_t> Tokenizer::encode_chat(const std::string& system_prompt, const std::string& user_prompt) const {
    std::vector<int32_t> tokens;
    if (bos_token_id >= 0) tokens.push_back(bos_token_id);
    auto add_sect = [&](const std::string& role, const std::string& content) {
        if (start_header_id >= 0) tokens.push_back(start_header_id);
        auto r = encode(role); tokens.insert(tokens.end(), r.begin(), r.end());
        if (end_header_id >= 0) tokens.push_back(end_header_id);
        auto nl = encode("\n\n"); tokens.insert(tokens.end(), nl.begin(), nl.end());
        auto c = encode(content); tokens.insert(tokens.end(), c.begin(), c.end());
        if (eot_token_id >= 0) tokens.push_back(eot_token_id);
    };
    if (!system_prompt.empty()) add_sect("system", system_prompt);
    add_sect("user", user_prompt);
    if (start_header_id >= 0) tokens.push_back(start_header_id);
    auto asst = encode("assistant"); tokens.insert(tokens.end(), asst.begin(), asst.end());
    if (end_header_id >= 0) tokens.push_back(end_header_id);
    auto nl = encode("\n\n"); tokens.insert(tokens.end(), nl.begin(), nl.end());
    printf("[Tokenizer] Chat prompt: %zu tokens\n", tokens.size());
    return tokens;
}

Overwriting src/tokenizer.cpp


### `model.h`

In [134]:
%%writefile src/model.h
#pragma once
#include "gguf_loader.h"
#include "ggml.h"
#include "ggml-backend.h"
#include <string>
#include <vector>

// =============================================================================
// Llama 3.1 Model Configuration (read from GGUF metadata)
// =============================================================================

struct LlamaConfig {
    int32_t  n_vocab       = 128256;   // vocabulary size
    int32_t  n_embd        = 4096;     // embedding dimension
    int32_t  n_layer       = 32;       // number of transformer layers
    int32_t  n_head        = 32;       // number of attention heads (Q)
    int32_t  n_head_kv     = 8;        // number of KV heads (GQA)
    int32_t  n_ff          = 14336;    // feed-forward hidden dimension
    int32_t  n_ctx         = 2048;     // max context length for KV cache
    float    rope_theta    = 500000.0f; // RoPE frequency base
    float    rms_norm_eps  = 1e-5f;    // RMSNorm epsilon
    int32_t  n_head_dim    = 128;      // head dimension (n_embd / n_head)
    int32_t  n_gqa         = 4;        // GQA ratio (n_head / n_head_kv)
};

// =============================================================================
// Per-layer weights
// =============================================================================

struct LlamaLayer {
    // Attention
    struct ggml_tensor* attn_norm;    // RMSNorm weight [n_embd]
    struct ggml_tensor* attn_q;       // Q projection [n_embd, n_embd]
    struct ggml_tensor* attn_k;       // K projection [n_embd, n_kv_dim]
    struct ggml_tensor* attn_v;       // V projection [n_embd, n_kv_dim]
    struct ggml_tensor* attn_output;  // Output projection [n_embd, n_embd]

    // FFN (SwiGLU)
    struct ggml_tensor* ffn_norm;     // RMSNorm weight [n_embd]
    struct ggml_tensor* ffn_gate;     // Gate projection [n_embd, n_ff]
    struct ggml_tensor* ffn_up;       // Up projection [n_embd, n_ff]
    struct ggml_tensor* ffn_down;     // Down projection [n_ff, n_embd]
};

// =============================================================================
// Full Llama Model
// =============================================================================

struct LlamaModel {
    LlamaConfig config;

    // Token embedding
    struct ggml_tensor* token_embd;   // [n_vocab, n_embd]

    // Transformer layers
    std::vector<LlamaLayer> layers;

    // Output
    struct ggml_tensor* output_norm;  // RMSNorm [n_embd]
    struct ggml_tensor* output;       // LM head [n_embd, n_vocab]

    // KV Cache (pre-allocated)
    struct ggml_tensor* kv_cache_k;   // [n_layer, n_ctx, n_head_kv * head_dim]
    struct ggml_tensor* kv_cache_v;   // [n_layer, n_ctx, n_head_kv * head_dim]

    // ggml contexts and backend
    struct ggml_context* ctx_weight;  // context for weight tensor metadata
    struct ggml_context* ctx_kv;      // context for KV cache metadata
    ggml_backend_t       backend;
    ggml_backend_buffer_t buf_weight; // buffer for model weights
    ggml_backend_buffer_t buf_kv;     // buffer for KV cache

    // Current KV cache position (number of tokens cached)
    int kv_used = 0;
};

// Load model config from GGUF metadata
LlamaConfig load_config(const GGUFFile& gguf);

// Initialize the model: create ggml tensors and load weights from GGUF
void load_model(LlamaModel& model, GGUFFile& gguf);

// Free all model resources
void free_model(LlamaModel& model);

// =============================================================================
// Forward pass
// =============================================================================

// Run forward pass for a sequence of tokens
// tokens: input token IDs
// n_tokens: number of tokens (>1 for prefill, 1 for decode)
// n_past: number of tokens already in KV cache
// Returns logits for the last token [n_vocab]
float* llama_forward(LlamaModel& model, const int32_t* tokens, int n_tokens, int n_past);

Overwriting src/model.h


### `model.cpp`

In [135]:
%%writefile src/model.cpp
#include "model.h"
#include <cmath>
#include <cstring>
#include <stdexcept>
#include <algorithm>
#include <vector>

LlamaConfig load_config(const GGUFFile& gguf) {
    LlamaConfig cfg;
    std::string arch = gguf.has_key("general.architecture") ? gguf.get_str("general.architecture") : "llama";
    auto key = [&](const std::string& k) { return arch + "." + k; };
    cfg.n_embd = (int32_t)gguf.get_u32(key("embedding_length"));
    cfg.n_layer = (int32_t)gguf.get_u32(key("block_count"));
    cfg.n_head = (int32_t)gguf.get_u32(key("attention.head_count"));
    cfg.n_head_kv = (int32_t)gguf.get_u32_or(key("attention.head_count_kv"), cfg.n_head);
    cfg.n_ff = (int32_t)gguf.get_u32(key("feed_forward_length"));
    cfg.rope_theta = gguf.get_f32_or(key("rope.freq_base"), 500000.0f);
    cfg.rms_norm_eps = gguf.get_f32_or(key("attention.layer_norm_rms_epsilon"), 1e-5f);
    if (gguf.has_key(key("vocab_size"))) cfg.n_vocab = (int32_t)gguf.get_u32(key("vocab_size"));
    else if (gguf.has_key("tokenizer.ggml.tokens")) cfg.n_vocab = (int32_t)gguf.get_arr("tokenizer.ggml.tokens").size();
    cfg.n_head_dim = cfg.n_embd / cfg.n_head;
    cfg.n_gqa = cfg.n_head / cfg.n_head_kv;
    printf("[Config] n_vocab=%d, n_embd=%d, n_layer=%d, n_head=%d, n_head_kv=%d, n_ff=%d, head_dim=%d, gqa=%d\n",
           cfg.n_vocab, cfg.n_embd, cfg.n_layer, cfg.n_head, cfg.n_head_kv, cfg.n_ff, cfg.n_head_dim, cfg.n_gqa);
    return cfg;
}

static const GGUFTensorInfo* find_tensor(const GGUFFile& gguf, const std::string& name) {
    for (const auto& t : gguf.tensors) if (t.name == name) return &t;
    return nullptr;
}

void load_model(LlamaModel& model, GGUFFile& gguf) {
    const auto& cfg = model.config;
    ggml_backend_load_all();
    model.backend = ggml_backend_init_best();
    printf("[Model] Backend: %s\n", ggml_backend_name(model.backend));

    int n_tensors = 3 + cfg.n_layer * 9;
    struct ggml_init_params params = { ggml_tensor_overhead() * (size_t)(n_tensors + 16) + ggml_graph_overhead(), nullptr, true };
    model.ctx_weight = ggml_init(params);

    auto create_tensor = [&](const std::string& name, std::initializer_list<int64_t> shape) -> struct ggml_tensor* {
        const GGUFTensorInfo* info = find_tensor(gguf, name);
        enum ggml_type type = info ? (enum ggml_type)(int)info->type : GGML_TYPE_F32;
        auto dims = std::vector<int64_t>(shape);
        struct ggml_tensor* t = (dims.size() == 1) ? ggml_new_tensor_1d(model.ctx_weight, type, dims[0]) : ggml_new_tensor_2d(model.ctx_weight, type, dims[0], dims[1]);
        ggml_set_name(t, name.c_str());
        return t;
    };

    model.token_embd = create_tensor("token_embd.weight", {cfg.n_embd, cfg.n_vocab});
    model.layers.resize(cfg.n_layer);
    for (int i = 0; i < cfg.n_layer; i++) {
        auto& l = model.layers[i]; std::string p = "blk." + std::to_string(i) + ".";
        l.attn_norm = create_tensor(p+"attn_norm.weight", {cfg.n_embd});
        l.attn_q = create_tensor(p+"attn_q.weight", {cfg.n_embd, cfg.n_embd});
        l.attn_k = create_tensor(p+"attn_k.weight", {cfg.n_embd, cfg.n_head_kv*cfg.n_head_dim});
        l.attn_v = create_tensor(p+"attn_v.weight", {cfg.n_embd, cfg.n_head_kv*cfg.n_head_dim});
        l.attn_output = create_tensor(p+"attn_output.weight", {cfg.n_embd, cfg.n_embd});
        l.ffn_norm = create_tensor(p+"ffn_norm.weight", {cfg.n_embd});
        l.ffn_gate = create_tensor(p+"ffn_gate.weight", {cfg.n_embd, cfg.n_ff});
        l.ffn_up = create_tensor(p+"ffn_up.weight", {cfg.n_embd, cfg.n_ff});
        l.ffn_down = create_tensor(p+"ffn_down.weight", {cfg.n_ff, cfg.n_embd});
    }
    model.output_norm = create_tensor("output_norm.weight", {cfg.n_embd});
    model.output = create_tensor("output.weight", {cfg.n_embd, cfg.n_vocab});

    model.buf_weight = ggml_backend_alloc_ctx_tensors(model.ctx_weight, model.backend);
    printf("[Model] Weight buffer: %.2f MB\n", (float)ggml_backend_buffer_get_size(model.buf_weight) / (1024*1024));

    auto load_t = [&](struct ggml_tensor* t) {
        const GGUFTensorInfo* info = find_tensor(gguf, ggml_get_name(t));
        if (!info) { fprintf(stderr, "[Model] Warning: tensor '%s' not found\n", ggml_get_name(t)); return; }
        std::vector<uint8_t> b(ggml_nbytes(t));
        read_tensor_data(gguf, *info, b.data(), b.size());
        ggml_backend_tensor_set(t, b.data(), 0, b.size());
    };
    printf("[Model] Loading weights...\n");
    load_t(model.token_embd);
    for (int i = 0; i < cfg.n_layer; i++) {
        auto& l = model.layers[i];
        load_t(l.attn_norm); load_t(l.attn_q); load_t(l.attn_k); load_t(l.attn_v); load_t(l.attn_output);
        load_t(l.ffn_norm); load_t(l.ffn_gate); load_t(l.ffn_up); load_t(l.ffn_down);
        if ((i + 1) % 8 == 0) printf("[Model]  ... loaded %d / %d layers\n", i + 1, cfg.n_layer);
    }
    load_t(model.output_norm); load_t(model.output);
    printf("[Model] All weights loaded!\n");

    // KV Cache
    int n_kv_dim = cfg.n_head_kv * cfg.n_head_dim;
    struct ggml_init_params kv_p = { ggml_tensor_overhead()*2 + 256, nullptr, true };
    model.ctx_kv = ggml_init(kv_p);
    model.kv_cache_k = ggml_new_tensor_3d(model.ctx_kv, GGML_TYPE_F32, n_kv_dim, cfg.n_ctx, cfg.n_layer);
    model.kv_cache_v = ggml_new_tensor_3d(model.ctx_kv, GGML_TYPE_F32, n_kv_dim, cfg.n_ctx, cfg.n_layer);
    ggml_set_name(model.kv_cache_k, "kv_cache_k");
    ggml_set_name(model.kv_cache_v, "kv_cache_v");
    model.buf_kv = ggml_backend_alloc_ctx_tensors(model.ctx_kv, model.backend);
    size_t kv_bytes = ggml_nbytes(model.kv_cache_k);
    std::vector<uint8_t> zeros(kv_bytes, 0);
    ggml_backend_tensor_set(model.kv_cache_k, zeros.data(), 0, kv_bytes);
    ggml_backend_tensor_set(model.kv_cache_v, zeros.data(), 0, kv_bytes);
    printf("[Model] KV cache: %.2f MB per K/V\n", (float)kv_bytes / (1024*1024));
    model.kv_used = 0;
}

void free_model(LlamaModel& m) {
    if (m.buf_weight) ggml_backend_buffer_free(m.buf_weight);
    if (m.buf_kv) ggml_backend_buffer_free(m.buf_kv);
    if (m.ctx_weight) ggml_free(m.ctx_weight);
    if (m.ctx_kv) ggml_free(m.ctx_kv);
    if (m.backend) ggml_backend_free(m.backend);
}

// Debug function to print tokens as decoded strings
void debug_print_tokens(const char* label, const int32_t* tokens, int n_tokens, const std::vector<std::string>& vocab) {
    fprintf(stderr, "[Debug] %s (%d tokens): ", label, n_tokens);
    for (int i = 0; i < std::min(n_tokens, 30); i++) {
        int32_t id = tokens[i];
        if (id >= 0 && id < (int32_t)vocab.size()) {
            fprintf(stderr, "[%d]='%s' ", id, vocab[id].c_str());
        } else {
            fprintf(stderr, "[%d]=INVALID ", id);
        }
    }
    if (n_tokens > 30) fprintf(stderr, "... (truncated)");
    fprintf(stderr, "\n");
    fflush(stderr);
}

float* llama_forward(LlamaModel& model, const int32_t* tokens, int n_tokens, int n_past, const std::vector<std::string>* vocab_debug) {
    const auto& cfg = model.config;
    int n_kv_dim = cfg.n_head_kv * cfg.n_head_dim;
    int n_kv = n_past + n_tokens;

    fprintf(stderr, "[Forward] n_tokens=%d n_past=%d n_kv=%d\n", n_tokens, n_past, n_kv);
    fflush(stderr);

    // Build computation graph
    size_t buf_size = ggml_tensor_overhead() * (size_t)(512 + cfg.n_layer * 64)
                    + ggml_graph_overhead_custom(16384, false);
    std::vector<uint8_t> compute_buf(buf_size);
    struct ggml_init_params mp = { buf_size, compute_buf.data(), true };
    struct ggml_context* ctx = ggml_init(mp);
    struct ggml_cgraph* gf = ggml_new_graph_custom(ctx, 16384, false);

    // Input tensors
    struct ggml_tensor* inp = ggml_new_tensor_1d(ctx, GGML_TYPE_I32, n_tokens);
    ggml_set_name(inp, "inp_tokens");
    ggml_set_input(inp);

    struct ggml_tensor* pos = ggml_new_tensor_1d(ctx, GGML_TYPE_I32, n_tokens);
    ggml_set_name(pos, "inp_pos");
    ggml_set_input(pos);

    struct ggml_tensor* KQ_mask = ggml_new_tensor_2d(ctx, GGML_TYPE_F32, n_kv, n_tokens);
    ggml_set_name(KQ_mask, "KQ_mask");
    ggml_set_input(KQ_mask);

    // DEBUG: Print token info
    fprintf(stderr, "[Forward] DEBUG: input tokens[0]=%d tokens[1]=%d\n", tokens[0], n_tokens > 1 ? tokens[1] : -1);
    if (vocab_debug && !vocab_debug->empty()) {
        debug_print_tokens("Input tokens", tokens, n_tokens, *vocab_debug);
    }
    fflush(stderr);

    // Get token embeddings - use ggml_get_rows for embedding lookup
    struct ggml_tensor* cur = ggml_get_rows(ctx, model.token_embd, inp);
    fprintf(stderr, "[Forward] DEBUG: token_embd shape: ne[0]=%ld ne[1]=%ld type=%d\n",
            model.token_embd->ne[0], model.token_embd->ne[1], (int)model.token_embd->type);
    fprintf(stderr, "[Forward] DEBUG: cur shape after get_rows: ne[0]=%ld ne[1]=%ld\n",
            cur->ne[0], cur->ne[1]);
    fflush(stderr);

    for (int il = 0; il < cfg.n_layer; il++) {
        auto& l = model.layers[il];
        struct ggml_tensor* residual = cur;

        cur = ggml_rms_norm(ctx, cur, cfg.rms_norm_eps);
        cur = ggml_mul(ctx, cur, l.attn_norm);

        struct ggml_tensor* Q = ggml_mul_mat(ctx, l.attn_q, cur);
        struct ggml_tensor* K = ggml_mul_mat(ctx, l.attn_k, cur);
        struct ggml_tensor* V = ggml_mul_mat(ctx, l.attn_v, cur);

        Q = ggml_reshape_3d(ctx, Q, cfg.n_head_dim, cfg.n_head, n_tokens);
        K = ggml_reshape_3d(ctx, K, cfg.n_head_dim, cfg.n_head_kv, n_tokens);
        V = ggml_reshape_3d(ctx, V, cfg.n_head_dim, cfg.n_head_kv, n_tokens);

        Q = ggml_rope_ext(ctx, Q, pos, nullptr,
                           cfg.n_head_dim, 0, 0,
                           cfg.rope_theta, 1.0f, 0.0f, 1.0f, 0.0f, 0.0f);
        K = ggml_rope_ext(ctx, K, pos, nullptr,
                           cfg.n_head_dim, 0, 0,
                           cfg.rope_theta, 1.0f, 0.0f, 1.0f, 0.0f, 0.0f);

        struct ggml_tensor* K_flat = ggml_reshape_2d(ctx, K, n_kv_dim, n_tokens);
        struct ggml_tensor* V_flat = ggml_reshape_2d(ctx, V, n_kv_dim, n_tokens);

        struct ggml_tensor* k_cache_view = ggml_view_2d(ctx, model.kv_cache_k,
            n_kv_dim, n_tokens,
            model.kv_cache_k->nb[1],
            (il * cfg.n_ctx + n_past) * model.kv_cache_k->nb[1]);

        struct ggml_tensor* v_cache_view = ggml_view_2d(ctx, model.kv_cache_v,
            n_kv_dim, n_tokens,
            model.kv_cache_v->nb[1],
            (il * cfg.n_ctx + n_past) * model.kv_cache_v->nb[1]);

        ggml_build_forward_expand(gf, ggml_cpy(ctx, K_flat, k_cache_view));
        ggml_build_forward_expand(gf, ggml_cpy(ctx, V_flat, v_cache_view));

        struct ggml_tensor* K_full = ggml_reshape_3d(ctx,
            ggml_view_2d(ctx, model.kv_cache_k,

[Main] model=Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
[Main] prompt="Hello, world! How are you?" (26 chars)
[Main] test_load=0 test_tokenize=1
[Debug] GGUF Version: 3, Tensors: 292, Metadata KV: 33
[Debug] Tensor data offset: 7840928
[Tokenizer] Loaded 280147 BPE merges
[Tokenizer] ASCII chars in vocab: 94/95
[Tokenizer] Sample vocab: [0]='!' [1]='"' [2]='#'
[Tokenizer] Lookup: 'H'=39, 'e'=68
[Tokenizer] Loaded vocab (size=128256). Special IDs: BOS=128000, EOS=128009, EOT=128009
[Main] Testing tokenizer with: "Hello, world! How are you?"
Tokens (8): 9906 11 14957 0 4438 548 9514 30
Decoded: Hello,world!Howareyou?

                n_kv_dim, n_kv,
                model.kv_cache_k->nb[1],
                il * cfg.n_ctx * model.kv_cache_k->nb[1]),
            cfg.n_head_dim, cfg.n_head_kv, n_kv);

        struct ggml_tensor* V_full = ggml_reshape_3d(ctx,
            ggml_view_2d(ctx, model.kv_cache_v,
                n_kv_dim, n_kv,
                model.kv_cache_v->nb[1],
                il * cfg.n_ctx * model.kv_cache_v->nb[1]),
            cfg.n_head_dim, cfg.n_head_kv, n_kv);

        Q = ggml_permute(ctx, Q, 0, 2, 1, 3);
        K_full = ggml_permute(ctx, K_full, 0, 2, 1, 3);

        if (cfg.n_gqa > 1) {
            struct ggml_tensor* Q_shape = ggml_new_tensor_3d(ctx, K_full->type,
                cfg.n_head_dim, n_kv, cfg.n_head);
            K_full = ggml_repeat(ctx, K_full, Q_shape);
        }

        struct ggml_tensor* KQ = ggml_mul_mat(ctx, K_full, Q);
        KQ = ggml_scale(ctx, KQ, 1.0f / sqrtf((float)cfg.n_head_dim));
        KQ = ggml_add(ctx, KQ, KQ_mask);
        KQ = ggml_soft_max(ctx, KQ);

        V_full = ggml_cont(ctx, ggml_permute(ctx, V_full, 1, 2, 0, 3));
        if (cfg.n_gqa > 1) {
            struct ggml_tensor* V_target = ggml_new_tensor_3d(ctx, V_full->type,
                n_kv, cfg.n_head_dim, cfg.n_head);
            V_full = ggml_repeat(ctx, V_full, V_target);
        }

        struct ggml_tensor* KQV = ggml_mul_mat(ctx, V_full, KQ);
        KQV = ggml_permute(ctx, KQV, 0, 2, 1, 3);
        KQV = ggml_cont_2d(ctx, KQV, cfg.n_embd, n_tokens);

        cur = ggml_mul_mat(ctx, l.attn_output, KQV);
        cur = ggml_add(ctx, cur, residual);

        residual = cur;
        cur = ggml_rms_norm(ctx, cur, cfg.rms_norm_eps);
        cur = ggml_mul(ctx, cur, l.ffn_norm);

        struct ggml_tensor* gate = ggml_silu(ctx, ggml_mul_mat(ctx, l.ffn_gate, cur));
        struct ggml_tensor* up = ggml_mul_mat(ctx, l.ffn_up, cur);
        cur = ggml_mul_mat(ctx, l.ffn_down, ggml_mul(ctx, gate, up));
        cur = ggml_add(ctx, cur, residual);
    }

    cur = ggml_rms_norm(ctx, cur, cfg.rms_norm_eps);
    cur = ggml_mul(ctx, cur, model.output_norm);
    cur = ggml_mul_mat(ctx, model.output, cur);

    ggml_set_name(cur, "logits");
    ggml_set_output(cur);
    ggml_build_forward_expand(gf, cur);

    fprintf(stderr, "[Forward] Graph built: %d nodes\n", ggml_graph_n_nodes(gf));
    fprintf(stderr, "[Forward] DEBUG: cur tensor: ne[0]=%ld ne[1]=%ld nb[0]=%ld nb[1]=%ld\n",
            cur->ne[0], cur->ne[1], (long)cur->nb[0], (long)cur->nb[1]);
    fflush(stderr);

    // Allocate and compute using gallocr (simpler than scheduler)
    ggml_gallocr_t allocr = ggml_gallocr_new(ggml_backend_get_default_buffer_type(model.backend));
    if (!ggml_gallocr_alloc_graph(allocr, gf)) {
        fprintf(stderr, "[Forward] ERROR: Failed to allocate graph\n");
        ggml_gallocr_free(allocr);
        ggml_free(ctx);
        return nullptr;
    }

    fprintf(stderr, "[Forward] Graph allocated, setting inputs...\n");
    fflush(stderr);

    // Set input data
    ggml_backend_tensor_set(inp, tokens, 0, n_tokens * sizeof(int32_t));

    std::vector<int32_t> pos_vals(n_tokens);
    for (int i = 0; i < n_tokens; i++) pos_vals[i] = n_past + i;
    ggml_backend_tensor_set(pos, pos_vals.data(), 0, n_tokens * sizeof(int32_t));

    std::vector<float> mask_data(n_kv * n_tokens, 0.0f);
    for (int i = 0; i < n_tokens; i++) {
        for (int j = 0; j < n_kv; j++) {
            if (j > n_past + i) {
                mask_data[i * n_kv + j] = -INFINITY;
            }
        }
    }
    ggml_backend_tensor_set(KQ_mask, mask_data.data(), 0, mask_data.size() * sizeof(float));

    fprintf(stderr, "[Forward] Computing graph...\n");
    fflush(stderr);

    enum ggml_status status = ggml_backend_graph_compute(model.backend, gf);
    fprintf(stderr, "[Forward] Compute done, status=%d\n", (int)status);
    fflush(stderr);

    // Extract logits
    static std::vector<float> logits_buf;
    logits_buf.resize(cfg.n_vocab);

    size_t logits_offset;
    if (n_tokens == 1) {
        logits_offset = 0;
    } else {
        logits_offset = (size_t)(n_tokens - 1) * cur->nb[1];
    }

    ggml_backend_tensor_get(cur, logits_buf.data(), logits_offset, cfg.n_vocab * sizeof(float));

    // Debug: find top-5 logits
    std::vector<std::pair<float, int>> top_logits;
    for (int i = 0; i < cfg.n_vocab; i++) {
        top_logits.push_back({logits_buf[i], i});
    }
    std::partial_sort(top_logits.begin(), top_logits.begin() + 5, top_logits.end(),
                      [](auto& a, auto& b) { return a.first > b.first; });
    fprintf(stderr, "[Forward] DEBUG: Top 5 logits: ");
    for (int i = 0; i < 5; i++) {
        const char* decoded = vocab_debug && top_logits[i].second < (int)vocab_debug->size()
            ? (*vocab_debug)[top_logits[i].second].c_str() : "???";
        fprintf(stderr, "[id=%d='%s']=%.4f ", top_logits[i].second, decoded, top_logits[i].first);
    }
    fprintf(stderr, "\n");
    fflush(stderr);

    ggml_gallocr_free(allocr);
    ggml_free(ctx);

    fprintf(stderr, "[Forward] Done.\n");
    fflush(stderr);

    return logits_buf.data();
}

Overwriting src/model.cpp


### `main.cpp`

In [136]:
%%writefile src/main.cpp
#include "gguf_loader.h"
#include "tokenizer.h"
#include "model.h"
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <sstream>

void print_usage(char* prog) {
    printf("Usage: %s <model_path> [prompt] [options]\n", prog);
    printf("Options:\n");
    printf("  --max-tokens N       Max tokens to generate (default: 128)\n");
    printf("  --temp T             Sampling temperature (default: 0.7)\n");
    printf("  --system S          System prompt\n");
    printf("  --test-load         Just test loading the model\n");
    printf("  --test-tokenize T   Test tokenizer with text T\n");
    printf("  --prompt-tokens IDS Use pre-computed token IDs (comma-separated)\n");
}

int main(int argc, char** argv) {
    if (argc < 2) { print_usage(argv[0]); return 1; }
    std::string model_path = argv[1];
    std::string prompt = "";
    int max_tokens = 128;
    float temp = 0.7f;
    std::string system_prompt = "";
    bool test_load = false, test_tokenize = false;
    std::vector<int32_t> prompt_tokens_override;

    // Parse all arguments
    for (int i = 2; i < argc; i++) {
        std::string arg = argv[i];
        if (arg == "--max-tokens" && i+1 < argc) { max_tokens = std::stoi(argv[++i]); }
        else if (arg == "--temp" && i+1 < argc) { temp = std::stof(argv[++i]); }
        else if (arg == "--system" && i+1 < argc) { system_prompt = argv[++i]; }
        else if (arg == "--test-load") { test_load = true; }
        else if (arg == "--test-tokenize") {
            test_tokenize = true;
            if (i+1 < argc && argv[i+1][0] != '-') prompt = argv[++i];
        }
        else if (arg == "--prompt-tokens" && i+1 < argc) {
            std::string tokens_str = argv[++i];
            std::stringstream ss(tokens_str);
            std::string token;
            while (std::getline(ss, token, ',')) {
                prompt_tokens_override.push_back(std::stoi(token));
            }
        }
        else if (arg[0] != '-' && prompt.empty()) { prompt = arg; }
    }

    fprintf(stderr, "[Main] model=%s\n", model_path.c_str());
    fprintf(stderr, "[Main] prompt=\"%s\" (%zu chars)\n", prompt.c_str(), prompt.size());
    fprintf(stderr, "[Main] test_load=%d test_tokenize=%d prompt_tokens_override_size=%zu\n",
            test_load, test_tokenize, prompt_tokens_override.size());
    fflush(stderr);

    try {
        GGUFFile gguf = load_gguf(model_path.c_str());

        if (test_load) {
            fprintf(stderr, "[Main] Test load complete.\n");
            return 0;
        }

        Tokenizer tok;
        tok.load(gguf);

        // Early exit for tokenizer test - do NOT load model
        if (test_tokenize) {
            fprintf(stderr, "[Main] Testing tokenizer with: \"%s\"\n", prompt.c_str());
            auto ids = tok.encode(prompt);
            printf("Tokens (%zu): ", ids.size());
            for (int id : ids) printf("%d ", id);
            printf("\nDecoded: %s\n", tok.decode(ids).c_str());
            return 0;
        }

        // Load model
        LlamaModel model;
        model.config = load_config(gguf);
        load_model(model, gguf);

        // Use override tokens if provided, otherwise encode the prompt
        std::vector<int32_t> tokens;
        if (!prompt_tokens_override.empty()) {
            tokens = prompt_tokens_override;
            fprintf(stderr, "[Main] Using provided prompt tokens: %zu tokens\n", tokens.size());
        } else {
            if (prompt.empty()) {
                fprintf(stderr, "[Main] ERROR: No prompt provided.\n");
                print_usage(argv[0]);
                return 1;
            }
            tokens = tok.encode_chat(system_prompt, prompt);
        }

        fprintf(stderr, "[Main] Prompt tokens: %zu\n", tokens.size());
        fprintf(stderr, "[Main] First 10 token IDs: ");
        for (size_t i = 0; i < std::min(tokens.size(), (size_t)10); i++)
            fprintf(stderr, "%d ", tokens[i]);
        fprintf(stderr, "\n");
        fflush(stderr);

        printf("\n--- Generating ---\n");
        fflush(stdout);

        int n_past = 0;
        for (int i = 0; i < max_tokens; i++) {
            int n_eval = (n_past == 0) ? (int)tokens.size() : 1;
            fprintf(stderr, "[Main] Forward pass %d: n_eval=%d n_past=%d\n", i, n_eval, n_past);
            fflush(stderr);

            float* logits = llama_forward(model, tokens.data() + n_past, n_eval, n_past, &tok.vocab);

            if (!logits) {
                fprintf(stderr, "[Main] ERROR: llama_forward returned null\n");
                break;
            }

            // Check logits sanity
            float min_l = logits[0], max_l = logits[0];
            bool has_nan = false;
            for (int j = 0; j < model.config.n_vocab; j++) {
                if (std::isnan(logits[j]) || std::isinf(logits[j])) { has_nan = true; break; }
                if (logits[j] < min_l) min_l = logits[j];
                if (logits[j] > max_l) max_l = logits[j];
            }
            fprintf(stderr, "[Main] Logits: min=%.4f max=%.4f nan/inf=%s\n",
                    min_l, max_l, has_nan ? "YES" : "no");

            // Greedy sampling
            int32_t next_id = 0;
            float best_l = -1e30f;
            for (int j = 0; j < model.config.n_vocab; j++) {
                if (logits[j] > best_l) { best_l = logits[j]; next_id = j; }
            }

            fprintf(stderr, "[Main] Token %d: id=%d logit=%.4f text=\"%s\"\n",
                    i, next_id, best_l, tok.decode(next_id).c_str());
            fflush(stderr);

            if (next_id == tok.eot_token_id || next_id == tok.eos_token_id) {
                fprintf(stderr, "[Main] Stop: generated EOT/EOS token\n");
                break;
            }

            std::string piece = tok.decode(next_id);
            printf("%s", piece.c_str());
            fflush(stdout);

            tokens.push_back(next_id);
            n_past += n_eval;
        }
        printf("\n");
        fflush(stdout);
        fprintf(stderr, "[Main] Generation complete.\n");
        free_model(model);
    } catch (const std::exception& e) {
        fprintf(stderr, "[Main] EXCEPTION: %s\n", e.what());
        return 1;
    }
    return 0;
}

Overwriting src/main.cpp


In [137]:
%%shell
echo "=== Compiling Llama Inference Engine ==="

# Ensure we look in the right place for libraries
GGML_DIR="ggml"
GGML_INC="$GGML_DIR/include"
GGML_BUILD="$GGML_DIR/build/src"

# Find libraries specifically
LIBS="$GGML_BUILD/libggml.so $GGML_BUILD/libggml-base.so $GGML_BUILD/libggml-cpu.so"

# Compile
g++ -std=c++17 -O3 -DNDEBUG \
    -I$GGML_INC -I$GGML_DIR/src \
    src/gguf_loader.cpp \
    src/tokenizer.cpp \
    src/model.cpp \
    src/main.cpp \
    $LIBS \
    -lpthread -lm -ldl -fopenmp \
    -o llama_inference

echo "=== Build successful! ==="
ls -la llama_inference

=== Compiling Llama Inference Engine ===
src/model.cpp: In function ‘float* llama_forward(LlamaModel&, const int32_t*, int, int, const std::vector<std::__cxx11::basic_string<char> >*)’:
src/model.cpp:216:2: error: ‘Main’ was not declared in this scope
  216 | [Main] model=Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
      |  ^~~~
src/model.cpp: In lambda function:
src/model.cpp:216:8: error: expected ‘{’ before ‘model’
  216 | [Main] model=Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
      |        ^~~~~
src/model.cpp: In function ‘float* llama_forward(LlamaModel&, const int32_t*, int, int, const std::vector<std::__cxx11::basic_string<char> >*)’:
src/model.cpp:216:7: error: expected ‘)’ before ‘model’
  216 | [Main] model=Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
      |       ^~~~~~
      |       )
src/model.cpp:214:25: note: to match this ‘(’
  214 |             ggml_view_2d(ctx, model.kv_cache_k,
      |                         ^
src/model.cpp:219:26: error: ‘Tensors’ was not declared in this scope
  21

## Step 4: Download a GGUF Model

Download a quantized Llama 3.1 8B model. We use Q4_K_M (~4.9GB) which fits in Colab's free RAM.

> **Note**: You need a HuggingFace account with Llama 3.1 access.
> Alternatively, use any compatible GGUF file you have.

In [138]:
%%shell
# Option 1: Download from HuggingFace (requires huggingface-cli login)
# pip install -q huggingface_hub
# huggingface-cli download bartowski/Meta-Llama-3.1-8B-Instruct-GGUF \
#     Meta-Llama-3.1-8B-Instruct-Q8_0.gguf \
#     --local-dir . --local-dir-use-symlinks False

# Option 2: If you've already uploaded the model to Colab or Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/models/your-model.gguf .

# For now, set the model path:
MODEL_PATH="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
echo "Expected model at: $MODEL_PATH"
if [ -f "$MODEL_PATH" ]; then
    echo "Model found! Size: $(du -h $MODEL_PATH | cut -f1)"
else
    echo "Model not found. Please download or upload a GGUF model."
    echo ""
    echo "Quick download (uncomment above and login to HuggingFace):"
    echo "  pip install huggingface_hub"
    echo "  huggingface-cli login"
    echo "  huggingface-cli download bartowski/Meta-Llama-3.1-8B-Instruct-GGUF \\"
    echo "      Meta-Llama-3.1-8B-Instruct-Q8_0.gguf \\"
    echo "      --local-dir . --local-dir-use-symlinks False"
fi

Expected model at: Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
Model found! Size: 8.0G


In [139]:
!pip install -q huggingface_hub

In [140]:
import os
from huggingface_hub import hf_hub_download

# Downloading a GGUF file with CUDA-compatible quantization (Q8_0)
# Note: Q4_K_M is NOT CUDA-compatible - ggml-cuda doesn't support q4_K getrows operation
# Using Q8_0 instead (slightly larger but fully supported)
model_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
    local_dir=".",
    local_dir_use_symlinks=False
)

print(f"Model downloaded to: {os.path.abspath(model_path)}")

Model downloaded to: /content/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf


In [141]:
import os
model_name = 'Meta-Llama-3.1-8B-Instruct-Q8_0.gguf'
if os.path.exists(model_name):
    size = os.path.getsize(model_name)
    print(f'File size: {size / (1024**3):.2f} GB')
    with open(model_name, 'rb') as f:
        magic = f.read(4)
        print(f'Magic header: {magic} (Expected: b\'GGUF\')')
else:
    print('File does not exist.')

File size: 7.95 GB
Magic header: b'GGUF' (Expected: b'GGUF')


## Step 5: Test — Load Model & Print Info

First, let's verify GGUF loading works by reading the header and metadata.

In [142]:
%%shell
# Set the library path so the system can find libggml.so
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$(pwd)/ggml/build/src

# Use absolute path to ensure no directory confusion
MODEL_PATH="$(pwd)/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"

if [ ! -f "$MODEL_PATH" ]; then
    echo "ERROR: Model file '$MODEL_PATH' not found."
    exit 1
fi

echo "--- Testing Load with $MODEL_PATH ---"
./llama_inference "$MODEL_PATH" --test-load

--- Testing Load with /content/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf ---
[Main] model=/content/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
[Main] prompt="" (0 chars)
[Main] test_load=1 test_tokenize=0
[Debug] GGUF Version: 3, Tensors: 292, Metadata KV: 33
[Debug] Tensor data offset: 7840928
[Main] Test load complete.


## Step 6: Test — Tokenizer Round-Trip

Verify the BPE tokenizer can encode and decode text correctly.

In [143]:
%%shell
# Set the library path so the system can find libggml.so
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$(pwd)/ggml/build/src

MODEL_PATH="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
./llama_inference "$MODEL_PATH" --test-tokenize "Hello, world! How are you?"

[Main] model=Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
[Main] prompt="Hello, world! How are you?" (26 chars)
[Main] test_load=0 test_tokenize=1
[Debug] GGUF Version: 3, Tensors: 292, Metadata KV: 33
[Debug] Tensor data offset: 7840928
[Tokenizer] Loaded 280147 BPE merges
[Tokenizer] ASCII chars in vocab: 94/95
[Tokenizer] Sample vocab: [0]='!' [1]='"' [2]='#'
[Tokenizer] Lookup: 'H'=39, 'e'=68
[Tokenizer] Loaded vocab (size=128256). Special IDs: BOS=128000, EOS=128009, EOT=128009
[Main] Testing tokenizer with: "Hello, world! How are you?"
Tokens (8): 9906 11 14957 0 4438 548 9514 30 
Decoded: Hello,world!Howareyou?


In [144]:
!pip install -q tiktoken

In [ ]:
%%python
import subprocess
import tiktoken
import os

# Llama 3.1 uses cl100k_base encoding (same as GPT-4)
enc = tiktoken.get_encoding("cl100k_base")

prompt = "What is the meaning of life?"
tokens = enc.encode(prompt)
print(f"Prompt: {prompt}")
print(f"Token count: {len(tokens)}")
print(f"Token IDs: {tokens}")
print(f"Decoded (tiktoken): {enc.decode(tokens)}")

# Convert to comma-separated string for CLI
tokens_str = ",".join(map(str, tokens))
print(f"\nPassing tokens to inference engine: {tokens_str}")

## Step 7: Run Inference!

Now the main event — generate text with Llama 3.1 8B running entirely in C++.

In [ ]:
%%bash
export LD_LIBRARY_PATH="/content/ggml/build/src:/usr/local/cuda-12.8/compat:/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib"

echo "=== Testing with very short inference ==="
# Run just 3 tokens max to test
/content/llama_inference /content/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf "Hi" --max-tokens 3 2>&1 | tail -30

## Step 8: Try Your Own Prompts

Modify the prompt and parameters below to experiment.

In [ ]:
%%shell
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$(pwd)/ggml/build/src

MODEL_PATH="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

# Change these!
PROMPT="Explain quantum computing in simple terms."
TEMPERATURE=0.7
MAX_TOKENS=256
SYSTEM_PROMPT="You are a helpful, harmless, and honest AI assistant."

./llama_inference "$MODEL_PATH" "$PROMPT" \
    --temp $TEMPERATURE \
    --max-tokens $MAX_TOKENS \
    --system "$SYSTEM_PROMPT"


## Architecture Overview

```
Input tokens
    │
    ▼
┌──────────┐
│ Token     │  (Embedding lookup: 128,256 × 4,096)
│ Embedding │
└────┬─────┘
     │
     ▼  ×32 layers
┌────────────────────────────────────┐
│  RMSNorm                           │
│    ↓                               │
│  Q/K/V projections                 │
│    ↓                               │
│  RoPE (θ=500,000)                  │
│    ↓                               │
│  Grouped-Query Attention           │
│  (32 Q heads, 8 KV heads)         │
│  + KV Cache read/write            │
│    ↓                               │
│  Residual Add                      │
│    ↓                               │
│  RMSNorm                           │
│    ↓                               │
│  SwiGLU FFN                        │
│  gate=silu(x·W_gate)               │
│  out=(gate*x·W_up)·W_down         │
│    ↓                               │
│  Residual Add                      │
└────────────────────────────────────┘
     │
     ▼
┌──────────┐
│ RMSNorm  │
│ LM Head  │  → logits (128,256)
└──────────┘
     │
     ▼
  Sample → next token
```
